In [16]:
import os
import torch
import torch.nn as nn
from pathlib import Path
import xml.etree.ElementTree as ET
from PIL import Image
import numpy as np
import torchvision.transforms as transforms
import torch.nn.functional as F
import torch.optim as optim

cwd = Path(os.getcwd())
images_lib = cwd / "data" / "images" / "Images"
annotations_lib = cwd / "data" / "Annotation"

dtype = torch.float
device = torch.device("cuda:0")

In [17]:
# to do: transform label into an index label (0-119)
# labels input needs to be a tensor of (N, C) where C is # of labels

class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(3, 20, 3) 
        self.pool = nn.MaxPool2d(2, 2) 
        self.conv2 = nn.Conv2d(20, 16, 3)
        self.fc1 = nn.Linear(169280, 1000)
        self.fc2 = nn.Linear(1000, 300)
        self.fc3 = nn.Linear(300, 120)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 169280)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [18]:
class labeled_image:
    def __init__(self, image_path, meta_path):
        transform = transforms.Compose(
            [transforms.ToTensor(),
             transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])
        self.image = Image.open(str(image_path)+".jpg").resize((100,100))
        self.tensor = transform(self.image)
#         self.tensor = torch.tensor(np.array(Image.open(str(image_path)+".jpg")), device=device, dtype=dtype)
        self.meta = {node.tag: node.text for node in ET.parse(str(meta_path)).getroot().iter() }
        [self.meta.pop(key, None) for key in dict(self.meta) if '\n' in self.meta[key]]
        
def batch_generator(file_list, batch_size):
    n_total = len(file_list)
    n_images_left= len(file_list)
    n_images_used = 0
    assert n_images_left%batch_size == 0
    def _batch_generator():
        nonlocal n_total
        nonlocal n_images_left
        nonlocal n_images_used
        nonlocal batch_size
        if n_images_left == 0:
            return []
        else:
            n_images_left -= batch_size
            return_list = [labeled_image(file_list[n][0], file_list[n][1]) for n in range(n_images_used, n_images_used+batch_size)]
            n_images_used += batch_size
            return return_list
    return _batch_generator

def list_to_4tensor(tensor_list):
    b = torch.zeros(len(tensor_list), tensor_list[0].shape[0], tensor_list[0].shape[1], tensor_list[0].shape[2])
    for i in range(b.shape[0]):
        b[i] = tensor_list[i]
    return b

In [19]:
file_list = []
for directory in os.listdir(annotations_lib):
    for file in os.listdir(annotations_lib / directory):
        file_list += [(str(images_lib / directory / file), str(annotations_lib / directory / file))]
get_batch = batch_generator(file_list, 20)

In [20]:
net = Net()
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

In [25]:
trainloader = get_batch()
i=0
while len(trainloader) > 0:  # loop over the dataset multiple times

    running_loss = 0.0

    # get the inputs; data is a list of [inputs, labels]
    inputs = list_to_4tensor([data.tensor for data in trainloader])
    labels = [data.meta['name'] for data in trainloader]
    
    # zero the parameter gradients
    optimizer.zero_grad()

    # forward + backward + optimize
    outputs = net(inputs)
    print(outputs)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

    # print statistics
    running_loss += loss.item()
    print(i, running_list)
    trainloader = get_batch()
    i += 1
print('Finished Training')

tensor([[ 0.0196, -0.0030, -0.0375, -0.0590, -0.0758,  0.0540, -0.0220,  0.0303,
          0.0111, -0.0203,  0.0173, -0.0431, -0.0452,  0.0583,  0.0296, -0.0236,
         -0.0263, -0.0294,  0.0085, -0.0249,  0.0617, -0.0455,  0.0197,  0.0484,
         -0.0180,  0.0451,  0.0416,  0.0135, -0.0475, -0.0305, -0.0574, -0.0350,
          0.0201,  0.0467, -0.0403,  0.0449, -0.0550,  0.0171, -0.0137, -0.0354,
         -0.0213,  0.0385, -0.0537, -0.0175,  0.0553, -0.0074,  0.0041,  0.0203,
          0.0031, -0.0209,  0.0146,  0.0209, -0.0264, -0.0146,  0.0118,  0.0557,
         -0.0237, -0.0260,  0.0305, -0.0232,  0.0263, -0.0030,  0.0147,  0.0458,
          0.0388, -0.0253,  0.0378, -0.0331,  0.0028,  0.0266,  0.0085,  0.0125,
          0.0004, -0.0497,  0.0563, -0.0250, -0.0349,  0.0178,  0.0223, -0.0399,
          0.0506,  0.0134,  0.0713,  0.0354, -0.0411,  0.0014,  0.0166,  0.0209,
         -0.0043,  0.0035,  0.0718,  0.0304,  0.0309,  0.0154, -0.0171,  0.0348,
          0.0087,  0.0587, -

AttributeError: 'list' object has no attribute 'size'